# 🌲🌲🌲 Random Forest — versión mínima

**Idea:** en vez de fiarnos de **un solo árbol**, entrenamos **muchos árboles un poco distintos** y **los dejamos votar**. Gana lo que diga la mayoría.

Ejecuta cada celda con **Shift + Enter**, en orden.

## 1. Crear los datos
Esta celda fabrica `alumnos.csv` con 1.000 alumnos. **No hace falta entenderla.**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(12)
n = 1000
horas_estudio = np.clip(rng.gamma(4, 2.2, n), 0, 20).round(1)
asistencia = np.clip(rng.normal(80, 14, n), 40, 100).round(0)
nota_previa = np.clip(rng.normal(6, 1.8, n), 0, 10).round(1)
tareas = np.clip(np.round(asistencia / 12 + rng.normal(0, 1.5, n)), 0, 10)
z = (-7.2 + 0.18 * horas_estudio + 0.035 * asistencia + 0.35 * nota_previa + 0.12 * tareas
     + 2.5 * ((horas_estudio > 10) & (asistencia > 80))
     - 2.0 * (asistencia < 60)) * 2.0
aprueba = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

pd.DataFrame({"horas_estudio": horas_estudio, "asistencia": asistencia,
              "nota_previa": nota_previa, "tareas": tareas, "aprueba": aprueba}).to_csv("alumnos.csv", index=False)
print("✅ alumnos.csv creado")

## 2. Ver los datos
Cada fila es un alumno. `aprueba` vale 1 (aprueba) o 0 (suspende): es lo que queremos predecir.

In [ ]:
df = pd.read_csv("alumnos.csv")
df.head()

## 3. Separar entrenamiento (80 %) y examen (20 %)
Este modelo **no necesita escalar** los datos.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[["horas_estudio", "asistencia", "nota_previa", "tareas"]]
y = df["aprueba"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Entrenar y evaluar
Usamos un bosque de **200 árboles**.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

modelo = RandomForestClassifier(n_estimators=200, random_state=42)
modelo.fit(X_train, y_train)

prediccion = modelo.predict(X_test)
print("Aciertos en el examen:", accuracy_score(y_test, prediccion))
print("F1 aciertos en el examen:", f1_score(y_test, prediccion))

## 5. Un árbol solo frente al bosque

In [ ]:
from sklearn.tree import DecisionTreeClassifier

filas = []
for profundidad in range(1, 11):
    a = DecisionTreeClassifier(max_depth=profundidad, random_state=42).fit(X_train, y_train)
    filas.append({
        "profundidad": profundidad,
        "hojas": a.get_n_leaves(),
        "accuracy entrenamiento": round(accuracy_score(y_train, a.predict(X_train)), 3),
        "accuracy examen": round(accuracy_score(y_test, a.predict(X_test)), 3),
    })

tabla = pd.DataFrame(filas).set_index("profundidad")
depth_op = tabla["accuracy examen"].idxmax()

In [ ]:
arbol = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
arbolOpt = DecisionTreeClassifier(max_depth=depth_op, random_state=42).fit(X_train, y_train)

print("Un árbol solo → examen:", round(accuracy_score(y_test, arbol.predict(X_test)), 2))
print("Un árbol profundidad optima → examen:", round(accuracy_score(y_test, arbolOpt.predict(X_test)), 2))
print("Bosque        → examen:", round(accuracy_score(y_test, modelo.predict(X_test)), 2))

👀 El bosque acierta más que un árbol solo: al votar, **los errores de cada árbol se compensan**.

## 6. Predecir un alumno nuevo

In [ ]:
nuevo = pd.DataFrame({"horas_estudio": [12], "asistencia": [90], "nota_previa": [7.5], "tareas": [9]})

resultado = modelo.predict(nuevo)
print("Aprueba" if resultado[0] == 1 else "Suspende")

## Resumen
1. **Random Forest = muchos árboles distintos que votan.**
2. **No hace falta escalar.**
3. Funciona bien **sin ajustar casi nada**.

## ✍️ Ejercicios
1. Prueba `RandomForestClassifier(n_estimators=5)` en el paso 4. ¿Acierta más o menos?
2. Cambia los datos del alumno nuevo. ¿Cuándo pasa a "Suspende"?